In [249]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_log_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

In [250]:
path = './data/feature_engineered_pakwheels_data.csv'

In [251]:
df = pd.read_csv(path)
df.head()

,name,brand,year,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr,car_age,model,variant
0,Toyota Surf 1997 for Sale,Toyota,1997,100000,Petrol,Automatic,2700,Gujrat,4100000,29,Surf,NaN
1,Suzuki Jimny 1991 for Sale,Suzuki,1991,130000,Petrol,Manual,1000,Gujrat,1100000,35,Jimny,NaN
2,Honda City 2004 i-DSI Vario for Sale,Honda,2004,250000,Petrol,Automatic,1300,Lahore,1825000,22,City,i-DSI Vario
3,Honda City 2005 i-DSI for Sale,Honda,2005,250000,Petrol,Manual,1300,Lahore,1625000,21,City,i-DSI
4,Toyota IST 2003 1.3 A for Sale,Toyota,2003,100000,Petrol,Automatic,1300,Karachi,1500000,23,IST,1.3 A


### Preprocessing

In [252]:
print("Shape of dataset: ", df.shape)
print("Duplicate count: ", df.duplicated().sum())

Shape of dataset:  (24731, 12)
Duplicate count:  0


In [253]:
df = df.drop_duplicates().reset_index(drop=True)

In [254]:
for col in list(df.columns):
    print(f"{col}: ", df[col].isna().sum())

name:  0
brand:  0
year:  0
mileage_km:  0
fuel_type:  0
transmission:  0
engine_cc:  0
city:  0
price_pkr:  0
car_age:  0
model:  0
variant:  3629


In [255]:
df.drop(columns=['name', 'year'], inplace=True)
df.head()

,brand,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr,car_age,model,variant
0,Toyota,100000,Petrol,Automatic,2700,Gujrat,4100000,29,Surf,NaN
1,Suzuki,130000,Petrol,Manual,1000,Gujrat,1100000,35,Jimny,NaN
2,Honda,250000,Petrol,Automatic,1300,Lahore,1825000,22,City,i-DSI Vario
3,Honda,250000,Petrol,Manual,1300,Lahore,1625000,21,City,i-DSI
4,Toyota,100000,Petrol,Automatic,1300,Karachi,1500000,23,IST,1.3 A


In [256]:
df.shape

(24731, 10)

In [257]:
df['variant'] = df['variant'].fillna('standard')

In [258]:
print(df['variant'].isnull().sum())
print(df['variant'].nunique())
print(df['variant'].value_counts())

0
720
variant
standard                 3629
VXR                      2801
VX                       1268
EXi                       669
GA                        590
                         ... 
Luxel Premium Edition       1
RS 1.2                      1
Sahara                      1
Sport F Limited 1.6         1
13G                         1
Name: count, Length: 720, dtype: int64


Split data into train and test before scaling or encoding to prevent data leaks

In [259]:
df.head()

,brand,mileage_km,fuel_type,transmission,engine_cc,city,price_pkr,car_age,model,variant
0,Toyota,100000,Petrol,Automatic,2700,Gujrat,4100000,29,Surf,standard
1,Suzuki,130000,Petrol,Manual,1000,Gujrat,1100000,35,Jimny,standard
2,Honda,250000,Petrol,Automatic,1300,Lahore,1825000,22,City,i-DSI Vario
3,Honda,250000,Petrol,Manual,1300,Lahore,1625000,21,City,i-DSI
4,Toyota,100000,Petrol,Automatic,1300,Karachi,1500000,23,IST,1.3 A


In [260]:
X = df.drop(columns=['price_pkr'])
y = df['price_pkr']

In [261]:
X.head()

,brand,mileage_km,fuel_type,transmission,engine_cc,city,car_age,model,variant
0,Toyota,100000,Petrol,Automatic,2700,Gujrat,29,Surf,standard
1,Suzuki,130000,Petrol,Manual,1000,Gujrat,35,Jimny,standard
2,Honda,250000,Petrol,Automatic,1300,Lahore,22,City,i-DSI Vario
3,Honda,250000,Petrol,Manual,1300,Lahore,21,City,i-DSI
4,Toyota,100000,Petrol,Automatic,1300,Karachi,23,IST,1.3 A


In [262]:
y.head()

0    4100000
1    1100000
2    1825000
3    1625000
4    1500000
Name: price_pkr, dtype: int64

In [263]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, shuffle=True)

In [264]:
print("X_train: ", X_train.shape, "y_train: ", y_train.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)

X_train:  (19784, 9) y_train:  (19784,)
X_test:  (4947, 9) y_test:  (4947,)


Separarting numeric and categorical columns

In [265]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24731 entries, 0 to 24730
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         24731 non-null  object
 1   mileage_km    24731 non-null  int64 
 2   fuel_type     24731 non-null  object
 3   transmission  24731 non-null  object
 4   engine_cc     24731 non-null  int64 
 5   city          24731 non-null  object
 6   price_pkr     24731 non-null  int64 
 7   car_age       24731 non-null  int64 
 8   model         24731 non-null  object
 9   variant       24731 non-null  object
dtypes: int64(4), object(6)
memory usage: 1.9+ MB


In [266]:
numeric_cols = [col for col in X.columns if X[col].dtype == 'int64']
categoric_cols = [col for col in X.columns if X[col].dtype == 'object']
    
print(numeric_cols)
print(categoric_cols)
print()
print("X_train's shape: ", X_train.shape)
print('Total number of columns: ', len(numeric_cols)+len(categoric_cols))

['mileage_km', 'engine_cc', 'car_age']
['brand', 'fuel_type', 'transmission', 'city', 'model', 'variant']

X_train's shape:  (19784, 9)
Total number of columns:  9


Separarting low and high cardinality columns for further steps

In [267]:
for col in categoric_cols:
    print(f"{col}: ", df[col].nunique())

brand:  52
fuel_type:  6
transmission:  2
city:  279
model:  355
variant:  720


In [268]:
df['brand'].value_counts()

brand
Suzuki           8854
Toyota           6745
Honda            4683
Daihatsu         1352
Hyundai           659
Mitsubishi        622
Nissan            579
Mercedes Benz     449
KIA               149
Chevrolet          99
BMW                87
Jeep               74
Lexus              57
Daewoo             54
Mazda              49
Subaru             32
Land Rover         28
Range Rover        18
Chery              17
Volkswagen         11
Datsun             11
Changan            11
Audi               10
MINI                8
SsangYong           7
Sogo                6
Ford                6
Fiat                5
Isuzu               4
Proton              4
Hummer              4
Peugeot             3
Daehan              3
Adam                3
Chrysler            3
Jaguar              3
Porsche             3
Cadillac            2
FAW                 2
GMC                 2
Hino                2
Avatr               1
JAC                 1
Opel                1
Forthing            1
Alek

In [269]:
low_card_cols = [col for col in categoric_cols if df[col].nunique() < 60]
high_card_cols = [col for col in categoric_cols if df[col].nunique() >= 60]

print("low cardinality cols: ", low_card_cols)
print("high cardinality cols: ", high_card_cols)

low cardinality cols:  ['brand', 'fuel_type', 'transmission']
high cardinality cols:  ['city', 'model', 'variant']


### Evaluation helper fuction

In [270]:
results = {}

def evaluate(name, pipe):
    
    scoring = {
            "mae": "neg_mean_absolute_error",
            "mape": "neg_mean_absolute_percentage_error",
            "r2": "r2",
        }
    
    if "log" in name:
        scoring["rmsle"] = "neg_root_mean_squared_log_error"
    
    s = cross_validate(pipe, X_train, y_train, cv=5, scoring=scoring)
        
    results[name] = {
        "MAE_PKR": -s["test_mae"].mean(),
        "MAPE": -s["test_mape"].mean(),
        "RMSLE": -s["test_rmsle"].mean() if "test_rmsle" in s else np.nan,
        "R2": s["test_r2"].mean(),
    }
    
    return results[name]

## Raw Linear Regression

In [271]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.compose import TransformedTargetRegressor

# Making column transformer to automate preprocessing and avoid duplication of code plus data leaks
num_pipe = Pipeline([
    ('transformer', FunctionTransformer(np.log1p)),
    ('scaler', StandardScaler()),
])

low_card_pipe = Pipeline([
    ('ohe', OneHotEncoder(min_frequency=10, handle_unknown='infrequent_if_exist'))
])

high_card_pipe = Pipeline([
    ('target_enc', TargetEncoder(target_type='continuous', smooth='auto'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('low_card', low_card_pipe, low_card_cols),
    ('high_card', high_card_pipe, high_card_cols),
])


# Model 
lr_raw_pipe = Pipeline([
    ('prep', preprocessor),
    ('lr', LinearRegression())
])

# evaluation
evaluate('lr_raw_target', lr_raw_pipe)

{'MAE_PKR': 480727.5778908335,
 'MAPE': 0.3097463275368326,
 'RMSLE': nan,
 'R2': 0.8072789229012944}

### Logged LR
Applied np.log1 on target (price_pkr)

In [272]:
# prep
num_pipe = Pipeline([
    ('transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler())
])

low_card_pipe = Pipeline([
    ('ohe', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=10))
])

high_card_pipe = Pipeline([
    ('target_enc', TargetEncoder(smooth='auto', target_type='continuous')),
    ('transform', FunctionTransformer(np.log1p)),
    ('scale', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, numeric_cols),
    ('low_card', low_card_pipe, low_card_cols),
    ('high_card', high_card_pipe, high_card_cols),
])

# model
lr_log_pipe = Pipeline([
    ('prep', preprocessor),
    ('lr_log', TransformedTargetRegressor(
        regressor=LinearRegression(),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

# evaluation
evaluate('lr_log_target', lr_log_pipe)

{'MAE_PKR': 377101.01246286905,
 'MAPE': 0.18624728902461413,
 'RMSLE': 0.25480699630811704,
 'R2': 0.8444888395205409}

### Ridge

In [297]:
from sklearn.model_selection import GridSearchCV

ridge_pipe = Pipeline([
    ('prep', preprocessor),
    ('ridge', TransformedTargetRegressor(
        regressor=Ridge(),
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

grid = GridSearchCV(
    ridge_pipe, 
    {'ridge__regressor__alpha': np.logspace(-3, 3, 13)},
    scoring='neg_mean_absolute_percentage_error',
    cv=5, n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_estimator_)
evaluate('ridge_log', grid.best_estimator_)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('transform',
                                                                   FunctionTransformer(func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['mileage_km', 'engine_cc',
                                                   'car_age']),
                                                 ('low_card',
                                                  Pipeline(steps=[('ohe',
                                                                   OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                                 min_frequency=10))]),
                                                  ['brand', 'fu

{'MAE_PKR': 375770.69398428913,
 'MAPE': 0.1860187230418619,
 'RMSLE': 0.2551406386281466,
 'R2': 0.8450541598470341}

### Results

In [298]:
res = pd.DataFrame(results)
res.T

,MAE_PKR,MAPE,RMSLE,R2
lr_raw_target,480727.577891,0.309746,NaN,0.807279
lr_log_target,377101.012463,0.186247,0.254807,0.844489
ridge_log,375770.693984,0.186019,0.255141,0.845054


### Comparison

In [317]:
n = {}

for item, val in results.items():
    if item in ['lr_raw_target', 'lr_log_target']:
        n[item] = val
n

{'lr_raw_target': {'MAE_PKR': 480727.5778908335,
  'MAPE': 0.3097463275368326,
  'RMSLE': nan,
  'R2': 0.8072789229012944},
 'lr_log_target': {'MAE_PKR': 377101.01246286905,
  'MAPE': 0.18624728902461413,
  'RMSLE': 0.25480699630811704,
  'R2': 0.8444888395205409}}

In [318]:
# handler function
def compare(m1, m2):
    comparison_matrix = pd.DataFrame({k: results[k] for k in results if k in [m1, m2]})
    
    comparison_matrix["Absolute change"] = round(comparison_matrix[m2]-comparison_matrix[m1], 4)
    comparison_matrix["Relative change (%)"] = round(((comparison_matrix[m2]-comparison_matrix[m1])/comparison_matrix[m1])*100, 3)
    
    return comparison_matrix

In [319]:
compare("lr_raw_target", "lr_log_target")

,lr_raw_target,lr_log_target,Absolute change,Relative change (%)
MAE_PKR,480727.577891,377101.012463,-103626.5654,-21.556
MAPE,0.309746,0.186247,-0.1235,-39.871
RMSLE,NaN,0.254807,NaN,NaN
R2,0.807279,0.844489,0.0372,4.609


In [320]:
compare("lr_log_target", "ridge_log")

,lr_log_target,ridge_log,Absolute change,Relative change (%)
MAE_PKR,377101.012463,375770.693984,-1330.3185,-0.353
MAPE,0.186247,0.186019,-0.0002,-0.123
RMSLE,0.254807,0.255141,0.0003,0.131
R2,0.844489,0.845054,0.0006,0.067
